In [ ]:
import functools
import numpy as np
import qiskit as qk
import qiskit.providers.fake_provider as qk_fp
from scipy.integrate import simpson

import sys
sys.path.append("../")
import pulse_simulator as ps

import matplotlib.pyplot as plt

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()

dt = backend.configuration().dt * ns
duration = 225 # dt

# Can be acquired with backend.configuration().timing_constraints['granularity']
granularity = 16
GATE_TIME_SI_UNITS = duration * dt / ns

def get_closest_multiple_of(value, base_number):
    return int(value + base_number/2) - (int(value + base_number/2) % base_number)

def get_closest_multiple_of_16(num, granularity=16):
    return get_closest_multiple_of(num, granularity)

registers = [0, 1]

# Variables
# NOTE: If the Rabi rates are True, you have to calibrate!
config_vars = ps.backend_simulation_vars(backend, rabi=True, units=Units)

# Carrier frequencies of each control line
carriers = ps.backend_carriers(backend, config_vars)

config_vars

In [ ]:
from qiskit.circuit import Parameter
from qiskit import pulse

def parameterized_envelope(duration, parameter: Parameter):
    """Define a paramterized waveform.

    Arguments:
        duration -- Integer pulse length in terms of the the sampling period dt.
        parameter -- Qiskit circuit parameter.

    Returns:
        Qiskit zero pulse.
    """
    return pulse.Constant(duration, parameter)

In [ ]:
# library.discrete.c

In [ ]:
gates = {
    0: "x_red",
    1: "x_red",
}

gate_lookup = {"x_red": ps.gaussian_envelope(duration, dt=dt, angle=np.pi / 2)}

before = simpson(gate_lookup["x_red"].samples.real, dx=dt)
print(f"Before {before / np.pi} π")


drive_amp = qk.circuit.Parameter('drive_amp')
with qk.pulse.build(name="Current moment", backend=backend) as rabi_sched:
    for i, gate in gates.items():
        channel = qk.pulse.DriveChannel(i)
        samples = get_closest_multiple_of_16(
            qk.pulse.seconds_to_samples(GATE_TIME_SI_UNITS)
        )
        waveform = ps.custom_envelope(
            samples,
            ps.resample(samples, gate_lookup[gate].samples),
            dt=duration * dt / samples,
        )
        qk.pulse.play(parameterized_envelope(samples, drive_amp), channel)

# after = np.trapz(pulse_moment.instructions[0][1].pulse.samples.real, dx=dt)
# print(f"After {after / np.pi} π")
        
# after = np.trapz(pulse_moment.instructions[0][1].pulse.get_waveform().samples.real, dx=dt)
# print(f"After {after / np.pi} π")

In [ ]:
from sympy.abc import x as x_
x_ls = np.arange(0, duration).tolist()
y_ls = gate_lookup[gate].samples.real.tolist()
fit = sp.interpolating_spline(3, x_, x_ls, y_ls)

In [ ]:
numpy_fit = sp.lambdify(x_, fit, 'numpy')

In [ ]:
plt.plot(np.arange(0, duration), numpy_fit(np.array(np.arange(0, duration))))
plt.plot(np.arange(0, duration), gate_lookup[gate].samples.real)


In [ ]:
def scalable_symbolic_pulse(signal):
    """Create a symbolic pulse from a signal.

    Arguments:
        signal -- Signal to be converted to a symbolic pulse.

    Returns:
        Symbolic pulse.
    """
    return pulse.library.discrete.DiscreteApproximation(signal)


ScalableSymbolicPulse(
    pulse_type: str,
    duration: ParameterExpression | int,
    amp: ParameterValueType,
    angle: ParameterValueType,
    parameters: dict[str, ParameterExpression | complex] | None = None,
    name: str | None = None,
    limit_amplitude: bool | None = None,
    envelope: sym.Expr | None = None,
    constraints: sym.Expr | None = None,
    valid_amp_conditions: sym.Expr | None = None,
)

In [ ]:
rabi_gate = qk.circuit.Gate("rabi", 1, [drive_amp])
qc_rabi = qk.QuantumCircuit(1, 1)
qc_rabi.append(rabi_gate, [0])
qc_rabi.measure(0, 0)

# Register a low-level, custom pulse definition for the given gate.
qc_rabi.add_calibration(rabi_gate, (0,), rabi_sched, [drive_amp])

# Execute rabi circuits
drive_amps = np.linspace(.9, 1.1, 51)
exp_rabi_circs = [qc_rabi.assign_parameters({drive_amp: a / 2}, inplace=False) for a in drive_amps]

In [ ]:
rabi_gate = Gate("rabi", 1, [drive_amp])

qc_rabi = QuantumCircuit(1, 1)

qc_rabi.append(rabi_gate, [0])
qc_rabi.measure(0, 0)
qc_rabi.add_calibration(rabi_gate, (0,), rabi_sched, [drive_amp])

exp_rabi_circs = [qc_rabi.assign_parameters({drive_amp: a}, inplace=False) for a in drive_amps]

In [ ]:
# Check that resample works.
plt.step(np.linspace(0, duration, samples, endpoint=False), ps.resample(samples, gate_lookup[gate].samples).real)
plt.step(np.arange(0, duration), gate_lookup[gate].samples.real)